### Потоковая обработка Kafka: Лямбда- и Каппа-архитектура

Сравнение архитектур

* Лямбда-архитектура

Пакетный уровень: для повышения точности обрабатывает исторические данные пакетами.

Уровень стримов: Обрабатывает потоковую обработку в режиме реального времени с низкой задержкой

Уровень обслуживания: Объединяет пакетные и потоковые результаты для запросов

Плюсы: Обрабатывает данные, поступающие с опозданием, отказоустойчивые и точные результаты

Минусы: Сложность в обслуживании, дублирование логики, повышенные операционные издержки


* Архитектура Kappa

Потоковый уровень: Все данные обрабатываются по однопоточному конвейеру

Управление состоянием: Поддерживает состояние приложения в потоковых процессорах

Повторная обработка: Можно повторно обрабатывать весь набор данных из журналов Kafka

Плюсы: Более простая архитектура, единая кодовая база, простота в обслуживании

Минусы: Требует повторной обработки для внесения исправлений, более сложное управление состоянием


### Различия в архитектуре:

* Lambda: Уровни пакетной и потоковой обработки, сложный, но хорошо справляется с поздними данными
* Kappa: Обработка только потоками, более простой, но требующий повторной обработки для внесения исправлений

Используетcя брокер kafka для высокопроизводительной потоковой передачи
Имитировать обработку событий в реальном времени (пользовательские клики, транзакции)

* Lambda: Отдельные потребители пакетов и потоков

* Kappa: однопоточный процессор с управлением состоянием


Обе архитектуры будут обрабатывать **один и тот же источник данных** для сравнения

In [ ]:
%%sh
wget https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
tar -xzf kafka_2.13-3.7.0.tgz -C .
mv kafka_2.13-3.7.0 kafka
ls -l kafka
cd kafka

total 72
drwxr-xr-x 3 root root  4096 Feb  9  2024 bin
drwxr-xr-x 3 root root  4096 Feb  9  2024 config
drwxr-xr-x 2 root root 12288 Jan 27 17:42 libs
-rw-r--r-- 1 root root 15125 Feb  9  2024 LICENSE
drwxr-xr-x 2 root root  4096 Feb  9  2024 licenses
-rw-r--r-- 1 root root 28359 Feb  9  2024 NOTICE
drwxr-xr-x 2 root root  4096 Feb  9  2024 site-docs


--2026-01-27 17:41:55--  https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119028138 (114M) [application/x-gzip]
Saving to: ‘kafka_2.13-3.7.0.tgz’

     0K .......... .......... .......... .......... ..........  0%  133K 14m31s
    50K .......... .......... .......... .......... ..........  0% 1.72M 7m48s
   100K .......... .......... .......... .......... ..........  0%  267K 7m37s
   150K .......... .......... .......... .......... ..........  0% 6.47M 5m47s
   200K .......... .......... .......... .......... ..........  0%  331K 5m48s
   250K .......... .......... .......... .......... ..........  0% 1.84M 5m0s
   300K .......... .......... .......... .......... ..........  0% 10.3M 4m18s
   350K .......... .......... .......... .......... 

In [ ]:
import subprocess
import os
import time

current_dir = os.getcwd()
if not current_dir.endswith('/content'):
    os.chdir('/content')

kafka_dir = 'kafka'
# Correct the paths to be relative to the kafka_dir when cwd is set in Popen
zookeeper_script_relative = os.path.join('bin', 'zookeeper-server-start.sh')
zookeeper_config_relative = os.path.join('config', 'zookeeper.properties')

print(f"Starting ZooKeeper in the background from {os.getcwd()}/{kafka_dir}...")

try:
    # Use subprocess.Popen with os.setsid to completely detach the process
    # Redirect stdout and stderr to DEVNULL to prevent blocking by output
    process = subprocess.Popen(
        [zookeeper_script_relative, zookeeper_config_relative],
        cwd=kafka_dir,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        preexec_fn=os.setsid  # Detach the process from the current session
    )
    # Give ZooKeeper a moment to start up
    time.sleep(5)
    print(f"ZooKeeper process started with PID: {process.pid}. It is running in the background.")
    print("You can check its status using `!jps` or `!ps aux | grep zookeeper` in a new cell.")
except Exception as e:
    print(f"Error starting ZooKeeper: {e}")
    print("Please ensure Kafka is extracted to a 'kafka' directory in /content.")

Starting ZooKeeper in the background from /content/kafka...
ZooKeeper process started with PID: 1052. It is running in the background.
You can check its status using `!jps` or `!ps aux | grep zookeeper` in a new cell.


In [ ]:
!ps aux | grep zookeeper

root        1052 34.2  0.6 3127560 81168 ?       Ssl  17:42   0:01 java -Xmx512M -Xms512M -server -XX:+UseG1GC -XX:MaxGCPauseMillis=20 -XX:InitiatingHeapOccupancyPercent=35 -XX:+ExplicitGCInvokesConcurrent -XX:MaxInlineLevel=15 -Djava.awt.headless=true -Xlog:gc*:file=/content/kafka/bin/../logs/zookeeper-gc.log:time,tags:filecount=10,filesize=100M -Dcom.sun.management.jmxremote -Dcom.sun.management.jmxremote.authenticate=false -Dcom.sun.management.jmxremote.ssl=false -Dkafka.logs.dir=/content/kafka/bin/../logs -Dlog4j.configuration=file:bin/../config/log4j.properties -cp /content/kafka/bin/../libs/activation-1.1.1.jar:/content/kafka/bin/../libs/aopalliance-repackaged-2.6.1.jar:/content/kafka/bin/../libs/argparse4j-0.7.0.jar:/content/kafka/bin/../libs/audience-annotations-0.12.0.jar:/content/kafka/bin/../libs/caffeine-2.9.3.jar:/content/kafka/bin/../libs/checker-qual-3.19.0.jar:/content/kafka/bin/../libs/commons-beanutils-1.9.4.jar:/content/kafka/bin/../libs/commons-cli-1.4.jar:/content/

In [ ]:
%%sh
pip install confluent-kafka==2.3.0 kafka-python==2.0.2 -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
%%writefile config.py
KAFKA_CONFIG = {
    'bootstrap.servers': '127.0.0.1:9092',
    'group.id': 'stream_processing_demo',
    'auto.offset.reset': 'earliest'
}

TOPICS = {
    'raw_events': 'raw_events',
    'processed_batch': 'processed_batch',
    'processed_stream': 'processed_stream',
    'kappa_output': 'kappa_output'
}

Writing config.py


In [ ]:
%%writefile data_generator.py
import json
import time
import random
from datetime import datetime
from confluent_kafka import Producer
from config import KAFKA_CONFIG, TOPICS

class EventGenerator:
    def __init__(self):
        self.producer = Producer({'bootstrap.servers': KAFKA_CONFIG['bootstrap.servers']})

    def generate_event(self):
        return {
            'user_id': random.randint(1, 1000),
            'event_type': random.choice(['click', 'purchase', 'view', 'signup']),
            'amount': random.uniform(10, 500) if random.random() > 0.7 else 0,
            'timestamp': datetime.now().isoformat(),
            'session_id': random.randint(1, 100)
        }

    def produce_events(self, num_events=1000, delay=0.1):
        for i in range(num_events):
            event = self.generate_event()
            self.producer.produce(
                TOPICS['raw_events'],
                key=str(event['user_id']),
                value=json.dumps(event)
            )
            if i % 100 == 0:
                self.producer.flush()
                print(f"Создал {i} событий")
            time.sleep(delay)

        self.producer.flush()
        print(f"Создано {num_events} событий")

if __name__ == "__main__":
    generator = EventGenerator()
    generator.produce_events()

Writing data_generator.py


In [ ]:
%%writefile kappa_architecture.py
import json
import time
from collections import defaultdict
from datetime import datetime, timedelta
from confluent_kafka import Consumer, Producer
from config import KAFKA_CONFIG, TOPICS

class KappaArchitecture:
    def __init__(self):
        self.consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'kappa_processor'})
        self.producer = Producer({'bootstrap.servers': KAFKA_CONFIG['bootstrap.servers']})

        self.state_store = defaultdict(lambda: {
            'count': 0,
            'total_amount': 0,
            'events': [],
            'first_seen': None,
            'last_seen': None,
            'session_counts': defaultdict(int)
        })

        self.window_size = timedelta(minutes=1)
        self.last_checkpoint = datetime.now()

    def process_stream(self):
        self.consumer.subscribe([TOPICS['raw_events']])

        print("Потоковый процессор архитектуры Kappa запущен")

        while True:
            msg = self.consumer.poll(1.0)
            if msg is None:
                continue

            if msg.error():
                print(f"Ошибка потребителя: {msg.error()}")
                continue

            event = json.loads(msg.value().decode('utf-8'))
            self.process_event(event)

            current_time = datetime.now()
            if current_time - self.last_checkpoint >= self.window_size:
                self.emit_windowed_results()
                self.last_checkpoint = current_time

    def process_event(self, event):
        user_id = event['user_id']
        event_time = datetime.fromisoformat(event['timestamp'])

        user_state = self.state_store[user_id]

        user_state['count'] += 1
        user_state['total_amount'] += event['amount']
        user_state['events'].append(event['event_type'])
        user_state['session_counts'][event['session_id']] += 1

        if user_state['first_seen'] is None:
            user_state['first_seen'] = event_time
        user_state['last_seen'] = event_time

        real_time_result = {
            'user_id': user_id,
            'event_count': user_state['count'],
            'total_amount': user_state['total_amount'],
            'avg_amount': user_state['total_amount'] / user_state['count'],
            'unique_sessions': len(user_state['session_counts']),
            'latest_event_type': event['event_type'],
            'user_lifetime_seconds': (user_state['last_seen'] - user_state['first_seen']).total_seconds(),
            'processed_at': datetime.now().isoformat(),
            'architecture': 'kappa'
        }

        self.producer.produce(
            TOPICS['kappa_output'],
            key=str(user_id),
            value=json.dumps(real_time_result)
        )

        if user_state['count'] % 5 == 0:
            print(f"Kappa: Пользователь {user_id} - {user_state['count']} событий, ${user_state['total_amount']:.2f} всего")
            self.producer.flush()

    def emit_windowed_results(self):
        window_results = []

        for user_id, state in self.state_store.items():
            if state['count'] == 0:
                continue

            event_type_counts = defaultdict(int)
            for event_type in state['events']:
                event_type_counts[event_type] += 1

            windowed_result = {
                'user_id': user_id,
                'window_start': (self.last_checkpoint - self.window_size).isoformat(),
                'window_end': self.last_checkpoint.isoformat(),
                'total_events': state['count'],
                'total_amount': state['total_amount'],
                'avg_amount_per_event': state['total_amount'] / state['count'],
                'unique_sessions': len(state['session_counts']),
                'event_type_distribution': dict(event_type_counts),
                'most_common_event': max(event_type_counts.items(), key=lambda x: x[1])[0] if event_type_counts else None,
                'user_activity_span': (state['last_seen'] - state['first_seen']).total_seconds() if state['first_seen'] else 0,
                'events_per_session': {str(k): v for k, v in state['session_counts'].items()},
                'architecture': 'kappa_windowed'
            }

            window_results.append(windowed_result)

        if window_results:
            print(f"Kappa: Вывод оконных результатов для {len(window_results)} пользователей")
            for result in window_results:
                self.producer.produce(
                    TOPICS['kappa_output'],
                    key=str(result['user_id']),
                    value=json.dumps(result)
                )
            self.producer.flush()

    def reprocess_from_offset(self, offset=0):
        print(f"Kappa: Переобработка с оффсета {offset}")

        self.state_store.clear()

        reprocess_consumer = Consumer({
            **KAFKA_CONFIG,
            'group.id': 'kappa_reprocess',
            'auto.offset.reset': 'earliest'
        })
        reprocess_consumer.subscribe([TOPICS['raw_events']])

        processed_count = 0
        while processed_count < 1000:
            msg = reprocess_consumer.poll(1.0)
            if msg is None:
                break

            if msg.error():
                continue

            event = json.loads(msg.value().decode('utf-8'))
            self.process_event(event)
            processed_count += 1

            if processed_count % 100 == 0:
                print(f"Переобработано {processed_count} событий")

        reprocess_consumer.close()
        print(f"Kappa: Переобработка завершена, {processed_count} событий обработано")

    def run(self):
        print("Архитектура Kappa запущена...")

        try:
            self.process_stream()
        except KeyboardInterrupt:
            print("Архитектура Kappa остановлена")

if __name__ == "__main__":
    kappa_arch = KappaArchitecture()
    kappa_arch.run()

Writing kappa_architecture.py


In [ ]:
%%writefile lambda_architecture.py
import json
import time
import threading
from collections import defaultdict
from datetime import datetime, timedelta
from confluent_kafka import Consumer, Producer
from config import KAFKA_CONFIG, TOPICS

class LambdaArchitecture:
    def __init__(self):
        self.batch_consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'lambda_batch'})
        self.stream_consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'lambda_stream'})
        self.producer = Producer({'bootstrap.servers': KAFKA_CONFIG['bootstrap.servers']})

        self.batch_data = []
        self.stream_state = defaultdict(lambda: {'count': 0, 'total_amount': 0})
        self.batch_results = {}

    def batch_layer(self):
        self.batch_consumer.subscribe([TOPICS['raw_events']])
        batch_window = timedelta(minutes=5)
        last_batch_time = datetime.now()

        print("Слой Lambda для батча запущен")

        while True:
            msg = self.batch_consumer.poll(1.0)
            if msg is None:
                continue

            if msg.error():
                print(f"Ошибка обработки батча: {msg.error()}")
                continue

            event = json.loads(msg.value().decode('utf-8'))
            self.batch_data.append(event)

            current_time = datetime.now()
            if current_time - last_batch_time >= batch_window:
                self.process_batch()
                last_batch_time = current_time

    def process_batch(self):
        if not self.batch_data:
            return

        user_stats = defaultdict(lambda: {'count': 0, 'total_amount': 0, 'events': []})

        for event in self.batch_data:
            user_id = event['user_id']
            user_stats[user_id]['count'] += 1
            user_stats[user_id]['total_amount'] += event['amount']
            user_stats[user_id]['events'].append(event['event_type'])

        for user_id, stats in user_stats.items():
            batch_result = {
                'user_id': user_id,
                'batch_count': stats['count'],
                'batch_total_amount': stats['total_amount'],
                'batch_avg_amount': stats['total_amount'] / stats['count'] if stats['count'] > 0 else 0,
                'event_types': list(set(stats['events'])),
                'processed_at': datetime.now().isoformat(),
                'layer': 'batch'
            }

            self.batch_results[user_id] = batch_result
            self.producer.produce(
                TOPICS['processed_batch'],
                key=str(user_id),
                value=json.dumps(batch_result)
            )

        print(f"Слой батчей обработал {len(self.batch_data)} событий {len(user_stats)} пользователей")
        self.batch_data.clear()
        self.producer.flush()

    def speed_layer(self):
        self.stream_consumer.subscribe([TOPICS['raw_events']])

        print("Слой Lambda для стримов запущен")

        while True:
            msg = self.stream_consumer.poll(1.0)
            if msg is None:
                continue

            if msg.error():
                print(f"Ошибка обработки стрима: {msg.error()}")
                continue

            event = json.loads(msg.value().decode('utf-8'))
            user_id = event['user_id']

            self.stream_state[user_id]['count'] += 1
            self.stream_state[user_id]['total_amount'] += event['amount']

            stream_result = {
                'user_id': user_id,
                'stream_count': self.stream_state[user_id]['count'],
                'stream_total_amount': self.stream_state[user_id]['total_amount'],
                'stream_avg_amount': (self.stream_state[user_id]['total_amount'] /
                                    self.stream_state[user_id]['count']),
                'latest_event': event['event_type'],
                'processed_at': datetime.now().isoformat(),
                'layer': 'speed'
            }

            self.producer.produce(
                TOPICS['processed_stream'],
                key=str(user_id),
                value=json.dumps(stream_result)
            )

            if self.stream_state[user_id]['count'] % 10 == 0:
                print(f"Слой стримов : Пользователь {user_id} обработал {self.stream_state[user_id]['count']} событий")
                self.producer.flush()

    def serving_layer(self):
        batch_consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'lambda_serving_batch'})
        stream_consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'lambda_serving_stream'})

        batch_consumer.subscribe([TOPICS['processed_batch']])
        stream_consumer.subscribe([TOPICS['processed_stream']])

        print("Слой Lambda для обслуживания запущен")

        while True:
            batch_msg = batch_consumer.poll(0.1)
            stream_msg = stream_consumer.poll(0.1)

            if batch_msg and not batch_msg.error():
                batch_data = json.loads(batch_msg.value().decode('utf-8'))
                user_id = batch_data['user_id']

                combined_result = batch_data.copy()
                if user_id in self.stream_state:
                    stream_stats = self.stream_state[user_id]
                    combined_result.update({
                        'combined_count': batch_data['batch_count'] + stream_stats['count'],
                        'combined_total_amount': batch_data['batch_total_amount'] + stream_stats['total_amount']
                    })

                print(f"Слой Lambda для обслуживания: комбинированный результат для пользователя {user_id}: {combined_result}")

            if stream_msg and not stream_msg.error():
                stream_data = json.loads(stream_msg.value().decode('utf-8'))
                print(f"Слой Lambda для обслуживания: данные реального времени для пользователя {stream_data['user_id']}")

    def run(self):
        batch_thread = threading.Thread(target=self.batch_layer)
        speed_thread = threading.Thread(target=self.speed_layer)
        serving_thread = threading.Thread(target=self.serving_layer)

        batch_thread.daemon = True
        speed_thread.daemon = True
        serving_thread.daemon = True

        batch_thread.start()
        speed_thread.start()
        serving_thread.start()

        print("Запущена 3-х слойная архитектура Lambda...")

        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("Остановлена  3-х слойная архитектура Lambda...")

if __name__ == "__main__":
    lambda_arch = LambdaArchitecture()
    lambda_arch.run()


Writing lambda_architecture.py


In [ ]:
import threading
import time
from lambda_architecture import LambdaArchitecture
from kappa_architecture import KappaArchitecture
from data_generator import EventGenerator

class ArchitectureComparison:
    def __init__(self):
        self.lambda_arch = LambdaArchitecture()
        self.kappa_arch = KappaArchitecture()
        self.generator = EventGenerator()

    def run_comparison(self):
        print("Запуск демонстрации сравнения архитектур")
        print("=" * 50)

        generator_thread = threading.Thread(target=self.generator.produce_events, args=(500, 0.05))
        lambda_thread = threading.Thread(target=self.lambda_arch.run)
        kappa_thread = threading.Thread(target=self.kappa_arch.run)

        generator_thread.daemon = True
        lambda_thread.daemon = True
        kappa_thread.daemon = True

        print("Запуск генерации данных...")
        generator_thread.start()

        time.sleep(2)

        print("Запуск архитектуры Lambda...")
        lambda_thread.start()

        print("Запуск архитектуры Kappa...")
        kappa_thread.start()

        try:
            while True:
                time.sleep(10)
                print("\n" + "="*50)
                print("Lambda: Пакетная + потоковая обработка со слоем обслуживания")
                print("Kappa: Чисто потоковая обработка с управлением состоянием")
                print("Обе архитектуры обрабатывают один и тот же поток событий...")
                print("="*50 + "\n")
        except KeyboardInterrupt:
            print("\nДемонстрация сравнения остановлена")

if __name__ == "__main__":
    demo = ArchitectureComparison()
    demo.run_comparison()

Запуск демонстрации сравнения архитектур
Запуск генерации данных...
Запуск архитектуры Lambda...
Запуск архитектуры Kappa...
Слой Lambda для батча запущен
Архитектура Kappa запущена...
Потоковый процессор архитектуры Kappa запущен
Слой Lambda для стримов запущен
Слой Lambda для обслуживания запущен
Запущена 3-х слойная архитектура Lambda...
Создал 200 событий

Lambda: Пакетная + потоковая обработка со слоем обслуживания
Kappa: Чисто потоковая обработка с управлением состоянием
Обе архитектуры обрабатывают один и тот же поток событий...


Демонстрация сравнения остановлена


### Основные отличия:

* Сложность: Lambda имеет 3 отдельных уровня обработки по сравнению с одним потоком Kappa

* Задержка: Kappa обеспечивает меньшую задержку, в то время как Lambda имеет задержки при пакетной обработке

* Отказоустойчивость: Lambda лучше обрабатывает устаревшие данные, Kappa требует повторной обработки

* Обслуживание: Kappa проще в обслуживании, Lambda требует синхронизации нескольких кодовых баз

* Управление состоянием: Kappa поддерживает состояние в потоке, Lambda разделяет пакетное и потоковое состояния